# HyperKT — Stage 9E: Matched-budget DKT vs HyperKT and gated structural fusion

**Exploratory follow-up study, NOT final test evaluation.** Uses the same Stage 2 student-disjoint train/validation windows and the fixed Stage 3B learned hypergraph. Five paired seeds, three configurations (15 training runs). All three use **batch size 32**, AdamW 2e-4, weight decay 1e-4, max 20 epochs, patience 5, and validation AUC checkpoint selection. This controls batch size and training budget but **does not equalize parameter count or architecture**. Do not describe it as a definitive fair published-baseline comparison.

Models: (1) independently implemented DKT; (2) original Stage 8B HyperKT full model, retrained at batch 32; (3) HyperKT with a separately labeled learnable gated hypergraph residual. The gate is an exploratory architectural change. The five seeds do not vary Stage 3B hypergraph training. **Never load test_sequences.npz in this notebook.**

**GPU:** A100 or L4 preferred, T4 supported. Save to Drive after every improved epoch. Rerunning completed runs skips them.

In [ ]:
%pip -q install numpy pandas scipy scikit-learn matplotlib tqdm

In [ ]:
import io,os,json,zipfile,hashlib,random,gc,platform,time,shutil
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import roc_auc_score,log_loss,brier_score_loss,f1_score
from scipy import stats
from google.colab import files,drive
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',DEVICE,'GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 1. Upload Stage 2 and Stage 3B ZIPs and mount Drive

In [ ]:
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/HyperKT_Stage9E');ROOT.mkdir(parents=True,exist_ok=True)
uploaded=files.upload()
name2=next((k for k in uploaded if 'stage2' in k.lower()),None)
name3=next((k for k in uploaded if 'stage3b' in k.lower()),None)
assert name2 and name3,'Upload both Stage 2 and Stage 3B ZIP files.'
sha2=hashlib.sha256(uploaded[name2]).hexdigest();sha3=hashlib.sha256(uploaded[name3]).hexdigest()
z2=zipfile.ZipFile(io.BytesIO(uploaded[name2]));z3=zipfile.ZipFile(io.BytesIO(uploaded[name3]))
assert 'train_sequences.npz' in z2.namelist() and 'val_sequences.npz' in z2.namelist()
assert 'hypergraph_repaired.npz' in z3.namelist()
print('Stage 2 SHA256:',sha2,' | Stage 3B SHA256:',sha3)

In [ ]:
def read_npz(z,name):
    with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as a: return {k:a[k].copy() for k in a.files}
train=read_npz(z2,'train_sequences.npz'); val=read_npz(z2,'val_sequences.npz')
struct=read_npz(z3,'hypergraph_repaired.npz')
manifest=json.loads(z2.read('stage2_report.json'))
N_SKILLS=int(max(train['target_skill'].max(),val['target_skill'].max(),struct['skill_ids'].max())+1)
H=np.zeros((N_SKILLS,struct['assignments'].shape[1]),dtype=np.float32)
H[struct['skill_ids'].astype(int)]=struct['assignments'].astype(np.float32)
MAX_LEN=train['history_skill'].shape[1]
print('N_SKILLS:',N_SKILLS,'MAX_LEN:',MAX_LEN,'hyperedges:',H.shape[1])
assert N_SKILLS==151 and MAX_LEN==200, 'Unexpected source configuration: review before running.'


In [ ]:
assert train['mask'].sum()==222562 and val['mask'].sum()==23858
assert not (set(train['user_id'].tolist()) & set(val['user_id'].tolist())) if train['user_id'].ndim==1 else not (set(np.asarray(train['user_id']).ravel().tolist()) & set(np.asarray(val['user_id']).ravel().tolist()))
print('Train windows',len(train['mask']),'Val windows',len(val['mask']))

## 2. Dataset, model definitions, and causal checks

In [ ]:
class SeqDataset(Dataset):
    def __init__(self,d): self.d=d
    def __len__(self):return len(self.d['mask'])
    def __getitem__(self,i):
        return tuple(torch.as_tensor(self.d[k][i],dtype=torch.long if k in ('history_skill','history_correct','target_skill') else torch.float32) for k in ('history_skill','history_correct','target_skill','target_correct','mask'))
TRAIN_DS=SeqDataset(train); VAL_DS=SeqDataset(val)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
    try:torch.use_deterministic_algorithms(True,warn_only=True)
    except Exception:pass
class CausalBlock(nn.Module):
    def __init__(self,d=128,heads=8,drop=.1,decay=False):
        super().__init__();self.heads=heads;self.dhead=d//heads;self.decay=decay
        self.qkv=nn.Linear(d,3*d);self.proj=nn.Linear(d,d);self.drop=nn.Dropout(drop)
        self.ln1=nn.LayerNorm(d);self.ln2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Dropout(drop),nn.Linear(4*d,d))
        if decay:self.log_rate=nn.Parameter(torch.full((heads,),-3.0))
    def forward(self,x):
        B,T,D=x.shape; z=self.ln1(x)
        q,k,v=self.qkv(z).chunk(3,dim=-1)
        def split(t):return t.reshape(B,T,self.heads,self.dhead).transpose(1,2)
        q,k,v=map(split,(q,k,v))
        logits=torch.matmul(q,k.transpose(-2,-1))*(self.dhead**-.5)
        pos=torch.arange(T,device=x.device)
        dist=(pos[:,None]-pos[None,:]).clamp(min=0).float()
        if self.decay:logits=logits-torch.nn.functional.softplus(self.log_rate)[None,:,None,None]*dist[None,None,:,:]
        future=pos[None,:]>pos[:,None]
        logits=logits.masked_fill(future[None,None,:,:],torch.finfo(logits.dtype).min)
        a=torch.softmax(logits,dim=-1)
        x=x+self.drop(self.proj(torch.matmul(self.drop(a),v).transpose(1,2).reshape(B,T,D)))
        x=x+self.drop(self.ff(self.ln2(x)))
        return x
class HyperKT(nn.Module):
    def __init__(self,n_skills,H,hyper=False,decay=False,d=128,layers=2,heads=8,drop=.1):
        super().__init__();self.hyper=hyper
        self.skill=nn.Embedding(n_skills,d,padding_idx=1)
        self.response=nn.Embedding(2,d)
        self.pos=nn.Embedding(MAX_LEN,d)
        self.blocks=nn.ModuleList([CausalBlock(d,heads,drop,decay) for _ in range(layers)])
        self.norm=nn.LayerNorm(d)
        if hyper:
            self.register_buffer('assignments',torch.tensor(H))
            self.edge_embedding=nn.Parameter(torch.randn(H.shape[1],d)*.02)
            self.fusion=nn.Linear(d*2,d)
        self.output=nn.Sequential(nn.LayerNorm(d*2),nn.Linear(d*2,d),nn.GELU(),nn.Dropout(drop),nn.Linear(d,1))
    def forward(self,hs,hc,ts):
        B,T=hs.shape; pos=torch.arange(T,device=hs.device)[None,:]
        x=self.skill(hs)+self.response(hc.clamp(0,1))+self.pos(pos)
        for layer in self.blocks:x=layer(x)
        x=self.norm(x)
        target=self.skill(ts)
        if self.hyper:
            structural=self.assignments[ts]@self.edge_embedding
            target=target+torch.tanh(self.fusion(torch.cat([target,structural],dim=-1)))
        return self.output(torch.cat([x,target],dim=-1)).squeeze(-1)
VARIANTS={'DKT_matched':None,'HyperKT_matched':None,'HyperKT_gated':None}
SEEDS=[42,123,2024,3407,7777]


In [ ]:
class DKT(nn.Module):
    # Independent Stage 9C baseline implementation, not a certified official DKT reproduction.
    def __init__(self,n,d=128,dropout=.1):
        super().__init__();self.n=n;self.emb=nn.Embedding(n*2,d)
        self.rnn=nn.LSTM(d,d,batch_first=True);self.drop=nn.Dropout(dropout);self.out=nn.Linear(d,n)
    def forward(self,hs,hc,ts):
        h,_=self.rnn(self.emb(hs+hc.clamp(0,1)*self.n))
        return self.out(self.drop(h)).gather(-1,ts.unsqueeze(-1)).squeeze(-1)

class HyperKTGated(HyperKT):
    # Only change vs original HyperKT: a scalar learned gate on the hypergraph residual.
    def __init__(self,n_skills,H,d=128,layers=2,heads=8,drop=.1):
        super().__init__(n_skills,H,hyper=True,decay=True,d=d,layers=layers,heads=heads,drop=drop)
        self.hyper_gate=nn.Parameter(torch.tensor(-2.0))
    def forward(self,hs,hc,ts):
        B,T=hs.shape;pos=torch.arange(T,device=hs.device)[None,:]
        x=self.skill(hs)+self.response(hc.clamp(0,1))+self.pos(pos)
        for layer in self.blocks:x=layer(x)
        x=self.norm(x);target=self.skill(ts)
        structural=self.assignments[ts]@self.edge_embedding
        target=target+torch.sigmoid(self.hyper_gate)*torch.tanh(self.fusion(torch.cat([target,structural],dim=-1)))
        return self.output(torch.cat([x,target],dim=-1)).squeeze(-1)

def make_model(name):
    if name=='DKT_matched':return DKT(N_SKILLS).to(DEVICE)
    if name=='HyperKT_matched':return HyperKT(N_SKILLS,H,hyper=True,decay=True).to(DEVICE)
    if name=='HyperKT_gated':return HyperKTGated(N_SKILLS,H).to(DEVICE)
    raise ValueError(name)

# No future responses or target skills should influence earlier predictions.
seed_all(42)
for name in VARIANTS:
    m=make_model(name).eval()
    hs=torch.as_tensor(val['history_skill'][:2],device=DEVICE,dtype=torch.long)
    hc=torch.as_tensor(val['history_correct'][:2],device=DEVICE,dtype=torch.long)
    ts=torch.as_tensor(val['target_skill'][:2],device=DEVICE,dtype=torch.long)
    with torch.no_grad():
        base=m(hs,hc,ts)
        changed_hc=hc.clone();changed_ts=ts.clone();changed_hc[:,100:]=1-changed_hc[:,100:]
        changed_ts[:,100:]=(changed_ts[:,100:]+2)%N_SKILLS
        altered=m(hs,changed_hc,changed_ts)
        delta=(base[:,:100]-altered[:,:100]).abs().max().item()
    assert delta<1e-5,(name,delta)
    print(name,'causal max difference',delta,'parameters',sum(p.numel() for p in m.parameters()))
    del m
if torch.cuda.is_available():torch.cuda.empty_cache()

## 3. Matched training protocol and resumable experiments

In [ ]:
CFG=dict(batch_size=32,lr=2e-4,weight_decay=1e-4,max_epochs=20,patience=5,hidden=128,dropout=.1)
SEEDS=[42,123,2024,3407,7777]
protocol={'stage':'9E','type':'exploratory_matched_batch_and_budget','models':list(VARIANTS),'seeds':SEEDS,'config':CFG,'stage2_sha256':sha2,'stage3b_sha256':sha3,'test_evaluated':False,'cross_window_event_uniqueness':'unresolved','baseline_fidelity':'independent DKT implementation','hypergraph_seeds':'fixed Stage3B hypergraph'}
protocol_path=ROOT/'protocol.json'
if protocol_path.exists():
    old=json.loads(protocol_path.read_text())
    assert old==protocol,'Existing run has different protocol or input hashes; use another Drive folder.'
else:protocol_path.write_text(json.dumps(protocol,indent=2))
TRAIN_DS=SeqDataset(train);VAL_DS=SeqDataset(val)
print(json.dumps(protocol,indent=2))

In [ ]:
def metrics(y,p):
    y=np.asarray(y,dtype=np.int64);p=np.clip(np.asarray(p,dtype=np.float64),1e-7,1-1e-7)
    edges=np.linspace(0,1,16);inds=np.minimum(np.digitize(p,edges[1:-1]),14)
    ece=sum((inds==i).mean()*abs(y[inds==i].mean()-p[inds==i].mean()) for i in range(15) if np.any(inds==i))
    return dict(auc=float(roc_auc_score(y,p)),nll=float(log_loss(y,p,labels=[0,1])),brier=float(brier_score_loss(y,p)),f1=float(f1_score(y,p>=.5,zero_division=0)),ece15=float(ece),n=int(len(y)))
def loader(ds,shuffle,seed):
    g=torch.Generator().manual_seed(seed)
    return DataLoader(ds,batch_size=CFG['batch_size'],shuffle=shuffle,generator=g,num_workers=0,pin_memory=True)
def evaluate(model,dl):
    model.eval();Y=[];P=[]
    with torch.no_grad():
        for hs,hc,ts,y,mask in dl:
            hs,hc,ts=hs.to(DEVICE),hc.to(DEVICE),ts.to(DEVICE)
            p=torch.sigmoid(model(hs,hc,ts)).cpu().numpy(); m=mask.numpy().astype(bool)
            Y.append(y.numpy()[m]);P.append(p[m])
    return metrics(np.concatenate(Y),np.concatenate(P))


In [ ]:
def train_one_9e(seed,name):
    path=ROOT/f'{name}_seed{seed}';path.mkdir(exist_ok=True)
    if (path/'best.pt').exists() and (path/'best_metrics.json').exists() and (path/'complete.flag').exists():
        print('Skipping completed:',name,seed)
        return json.loads((path/'best_metrics.json').read_text())
    # Resume from epoch-level state if available, preserving optimizer, RNG, and loader order.
    state_path=path/'last_state.pt'
    seed_all(seed)
    model=make_model(name)
    opt=torch.optim.AdamW(model.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'])
    train_loader=loader(TRAIN_DS,True,seed)
    val_loader=loader(VAL_DS,False,seed)
    best=-1.;bad=0;history=[];start=1
    if state_path.exists():
        state=torch.load(state_path,map_location='cpu',weights_only=False)
        assert state['protocol']==protocol
        model.load_state_dict(state['model']);opt.load_state_dict(state['optimizer'])
        best=state['best'];bad=state['bad'];history=state['history'];start=state['epoch']+1
        torch.set_rng_state(state['torch_rng']);np.random.set_state(state['numpy_rng']);random.setstate(state['python_rng'])
        if torch.cuda.is_available() and state['cuda_rng'] is not None:torch.cuda.set_rng_state_all(state['cuda_rng'])
        train_loader.generator.set_state(state['loader_rng'])
        print('Resuming',name,seed,'at epoch',start)
    for epoch in range(start,CFG['max_epochs']+1):
        model.train();total=0.;count=0.
        for hs,hc,ts,y,mask in train_loader:
            hs,hc,ts,y,mask=[x.to(DEVICE) for x in (hs,hc,ts,y,mask)]
            opt.zero_grad(set_to_none=True)
            logits=model(hs,hc,ts)
            loss=(nn.functional.binary_cross_entropy_with_logits(logits,y,reduction='none')*mask).sum()/mask.sum().clamp(min=1)
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()
            total+=float(loss.item())*float(mask.sum());count+=float(mask.sum())
        v=evaluate(model,val_loader);history.append({'epoch':epoch,'train_nll':total/count,**v})
        print(f'{name} seed={seed} epoch={epoch} val_auc={v["auc"]:.6f}',flush=True)
        if v['auc']>best+1e-6:
            best=v['auc'];bad=0
            torch.save({'state_dict':{k:t.detach().cpu() for k,t in model.state_dict().items()},'seed':seed,'variant':name,'config':CFG,'best_epoch':epoch,'stage2_sha256':sha2,'stage3b_sha256':sha3},path/'best.pt')
            (path/'best_metrics.json').write_text(json.dumps({'seed':seed,'variant':name,'best_epoch':epoch,'n_parameters':sum(p.numel() for p in model.parameters()),**v},indent=2))
        else:bad+=1
        pd.DataFrame(history).to_csv(path/'history.csv',index=False)
        # Save epoch state atomically to recover from Colab interruptions.
        state={'protocol':protocol,'model':model.state_dict(),'optimizer':opt.state_dict(),'best':best,'bad':bad,'history':history,'epoch':epoch,'torch_rng':torch.get_rng_state(),'numpy_rng':np.random.get_state(),'python_rng':random.getstate(),'cuda_rng':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,'loader_rng':train_loader.generator.get_state()}
        temp=path/'last_state.tmp';torch.save(state,temp);os.replace(temp,state_path)
        if bad>=CFG['patience']:break
    (path/'complete.flag').write_text('complete')
    result=json.loads((path/'best_metrics.json').read_text())
    del model,opt;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return result

## 4. Execute 15 runs (can take considerable time; checkpoints persist in Drive)

In [ ]:
results=[]
for seed in SEEDS:
    for name in VARIANTS:
        results.append(train_one_9e(seed,name))
        pd.DataFrame(results).to_csv(ROOT/'partial_results.csv',index=False)
df=pd.DataFrame(results)
assert len(df)==15 and not df[['seed','variant']].duplicated().any()
df.to_csv(ROOT/'all_results.csv',index=False)
print(df[['variant','seed','auc','nll','brier','best_epoch']].to_string(index=False))

## 5. Statistical summary and transparent exploratory comparisons

In [ ]:
summary=df.groupby('variant')[['auc','nll','brier','f1','ece15']].agg(['mean','std'])
summary.to_csv(ROOT/'summary.csv');display(summary)
pivot=df.pivot(index='seed',columns='variant',values='auc')
rows=[]
for left,right in [('DKT_matched','HyperKT_matched'),('DKT_matched','HyperKT_gated'),('HyperKT_gated','HyperKT_matched')]:
    diff=(pivot[left]-pivot[right]).to_numpy();mean=float(diff.mean());sd=float(diff.std(ddof=1))
    ci=stats.t.interval(.95,len(diff)-1,loc=mean,scale=sd/np.sqrt(len(diff)))
    rows.append({'left':left,'right':right,'mean_auc_difference':mean,'sd':sd,'ci95_low':float(ci[0]),'ci95_high':float(ci[1]),'paired_t_p_exploratory':float(stats.ttest_rel(pivot[left],pivot[right]).pvalue),'n_seeds':len(diff)})
comparisons=pd.DataFrame(rows);comparisons.to_csv(ROOT/'paired_comparisons.csv',index=False);display(comparisons)
print('Caution: repeated seeds on one fixed validation cohort are not independent student samples; multiple comparisons exploratory.')

## 6. Export ZIP for review (model checkpoints stay on Drive)

In [ ]:
out=Path('/content/hyperkt_stage9e.zip')
with zipfile.ZipFile(out,'w',compression=zipfile.ZIP_DEFLATED) as zz:
    for f in ROOT.rglob('*'):
        if f.is_file() and f.suffix in ('.json','.csv','.flag'):
            zz.write(f,f.relative_to(ROOT))
print('Saved',out,'bytes',out.stat().st_size)
files.download(str(out))
print('Keep the original Drive folder containing best.pt and last_state.pt; the ZIP intentionally excludes large checkpoints.')